
# Incident Analysis With LLM

2. Выполняет предобработку через `IncidentRequestsPreprocessor`
3. Запускает два LLM-эксперимента на одной и той же таблице.


In [65]:

from functools import partial
from pathlib import Path
import json
import os
import re
import sys
from typing import Callable

import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub.utils import enable_progress_bars

ROOT = Path.cwd()
if not (ROOT / "incident_requests").exists() and (ROOT.parent / "incident_requests").exists():
    ROOT = ROOT.parent

sys.path.append(str(ROOT))

from incident_requests import IncidentRequestsPreprocessor

pd.set_option("display.max_colwidth", None)
enable_progress_bars()

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"


Легковесные модели: 
- Qwen/Qwen2.5-0.5B-Instruct
- Qwen/Qwen2.5-1.5B-Instruct
...


In [ ]:

INPUT_PATH = ROOT / "research/data" / "march_incidents.xlsx"
SHEET_NAME = 0

COLS2DROP = [
    "Источник",
    "Категория",
    "Комментарий к выполненным работам",
    "Статус заявки",
    "Исполнители", # эта и колонки ниже - пустые
    "Перечень материалов",
    "Услуги",
    "Стоимость",
    "Вложения",
]

TEXT_COLUMN = "Описание"
LEMMATIZED_TEXT_COLUMN = "Описание_леммы"
LLM_TEXT_COLUMN = TEXT_COLUMN
TYPE_COLUMN = "Тип инцидента"
OTHER_LABEL = "Прочее"
USE_NATASHA = True

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct" # Qwen/Qwen2.5-0.5B-Instruct

MODEL_CACHE_DIR = ROOT / ".cache" / "huggingface"
LOCAL_FILES_ONLY = False 
DEVICE_MAP = "auto"
MAX_NEW_TOKENS = 32


In [ ]:

def load_table(path: Path, sheet_name=0) -> pd.DataFrame:
    suffix = path.suffix.lower()

    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path, sheet_name=sheet_name)

    raise ValueError(f"Неподдерживаемый формат файла: {suffix}")


df = load_table(INPUT_PATH, sheet_name=SHEET_NAME)
processor = IncidentRequestsPreprocessor(
    df,
    columns_to_drop=COLS2DROP,
    detect_incident_type=False,
    use_natasha=USE_NATASHA,
)
preprocessed_df = processor.preprocess()
preprocessed_df[LEMMATIZED_TEXT_COLUMN] = preprocessed_df[TEXT_COLUMN].apply(processor._lemmatize_natasha)

print(preprocessed_df.dtypes)
display(preprocessed_df.drop(columns=[LEMMATIZED_TEXT_COLUMN], errors="ignore").head(2))


Дата                         datetime64[us]
Время                                   str
Адрес                                   str
Пом.                                    str
Подкатегория                            str
Описание                                str
Желаемое время выполнения               str
Дата исполнения               datetime64[s]
Координаторы                            str
Описание_леммы                          str
dtype: object


,Дата,Время,Адрес,Пом.,Подкатегория,Описание,Желаемое время выполнения,Дата исполнения,Координаторы
0,2026-03-15,23:19,"г Санкт-Петербург, п Парголово, ул Николая Рубцова, д. 5 стр. 1",309,Протечка,"СИЛЬНАЯ ТЕЧЬ СТОЯКА ГВС В ВАННОЙ, ОТКЛ. СТ. 29 ГВС В/З, ТРЕБ. ЗАМЕНА ОТСЕЧНОГО КРАНА, ВЫРВАЛО ШТОК\nГИЛЬДИЯ: Заменили два крана 1/2 на хгвс ст. 29 в/з\nЗапустили развоздушили",с 09:00 16.03.2026 по 13:00 16.03.2026,NaT,"ГИЛЬДИЯ СЕРВИСА ГИЛЬДИЯ СЕРВИСА СЕРВИСА, Бригадир санитарно-технических работ Журавский Антон Петрович"
1,2026-03-15,21:32,"г Санкт-Петербург, пр-кт Гладышевский, д. 38 корп. 2 стр. 1",267,Протечка,течь с потолка-ТЕЧЬ С КРОВЛИ,с 09:00 16.03.2026 по 15:00 16.03.2026,NaT,Начальник отделения Григорьев Игорь Валерьевич


In [69]:

def load_generation_model(
    model_name: str,
    cache_dir: Path | str | None = None,
    local_files_only: bool = False,
):
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        cache_dir=cache_dir,
        local_files_only=local_files_only,
        trust_remote_code=True,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        cache_dir=cache_dir,
        local_files_only=local_files_only,
        torch_dtype=torch_dtype,
        device_map=DEVICE_MAP,
        trust_remote_code=True,
    )
    model.eval()
    return tokenizer, model


def lemmatize(text: str) -> str:
    text = str(text).lower().replace("ё", "е")

    try:
        lemmatized_text = processor._lemmatize_natasha(text)
    except NameError:
        lemmatized_text = text

    normalized_text = lemmatized_text.lower().replace("ё", "е")
    normalized_text = re.sub(r"[^\w\s]", " ", normalized_text)
    return re.sub(r"\s+", " ", normalized_text).strip()


def parse_model_response(response_text: str, labels: list[str], other_label: str) -> str:
    """Extract and normalize the model response to one of the allowed labels."""
    def normalize_label(label: str) -> str:
        normalized_text = lemmatize(label)
        if not normalized_text:
            return other_label

        label_lookup = {lemmatize(label): label for label in labels}
        if normalized_text in label_lookup:
            return label_lookup[normalized_text]

        for normalized_label, original_label in label_lookup.items():
            if normalized_label in normalized_text:
                return original_label

        has_stoyak = "стояк" in normalized_text or re.search(r"\bст\b\s*\d*\s*(гвс|хвс|хгвс)", normalized_text)
        has_gvs = any(marker in normalized_text for marker in ("гвс", "хгвс", "горяч"))
        has_hvs = any(marker in normalized_text for marker in ("хвс", "хгвс", "холод"))
        has_pipe = "труб" in normalized_text or "трубопровод" in normalized_text
        has_leak = re.search(r"\b(теч|утеч|протеч)\w*", normalized_text)

        fallback_rules = [
            ("Лифты", "лифт" in normalized_text or "застрев" in normalized_text),
            ("Труба ГВС", has_pipe and has_gvs and not has_stoyak),
            ("Труба ХВС", has_pipe and has_hvs and not has_stoyak),
            ("Стояк ГВС", has_stoyak and has_gvs),
            ("Стояк ХВС", has_stoyak and has_hvs),
            ("Утечка стояка", has_stoyak and has_leak),
            ("Утечка радиатора", "радиатор" in normalized_text and has_leak),
        ]

        for normalized_label, condition in fallback_rules:
            if normalized_label in labels and condition:
                return normalized_label

        return other_label

    match = re.search(r"\{.*\}", response_text, flags=re.S)
    if match:
        try:
            payload = json.loads(match.group(0))
            label = str(payload.get("incident_type", "")).strip()
            normalized_label = normalize_label(label)
            if normalized_label in labels:
                return normalized_label
        except json.JSONDecodeError:
            pass

    cleaned_text = response_text.strip().strip('"')
    normalized_label = normalize_label(cleaned_text)
    if normalized_label in labels:
        return normalized_label

    for label in labels:
        if label.lower() in cleaned_text.lower():
            return label

    return other_label


def build_prompt(
    text: str,
    incident_types: list[str],
    prompt_instructions: str,
    other_label: str = OTHER_LABEL,
) -> str:
    labels = [*incident_types, other_label]
    options = "\n".join(f"- {label}" for label in labels)

    return f"""{prompt_instructions}

    Доступные типы инцидентов:
    {options}

    Текст заявки:
    {text}

    Верни только JSON без пояснений:
    {{"incident_type": "<один вариант из списка>"}}"""


def generate_incident_type(
    description: str,
    tokenizer,
    model,
    incident_types: list[str],
    prompt_builder: Callable[[str], str],
    other_label: str = OTHER_LABEL,
    return_raw: bool = False,
) -> str | dict[str, str]:
    if pd.isna(description) or not str(description).strip():
        if return_raw:
            return {"raw_response": "", "parsed_type": other_label}
        return other_label

    labels = [*incident_types, other_label]
    messages = [
        {
            "role": "system",
            "content": "Ты размечаешь заявки ЖКХ и возвращаешь только валидный JSON.",
        },
        {
            "role": "user",
            "content": prompt_builder(str(description)),
        },
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    model_inputs = tokenizer(prompt, return_tensors="pt")
    input_device = next(model.parameters()).device
    model_inputs = {key: value.to(input_device) for key, value in model_inputs.items()}

    with torch.inference_mode():
        generated = model.generate(
            **model_inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )

    new_tokens = generated[0][model_inputs["input_ids"].shape[1]:]
    response_text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    parsed_type = parse_model_response(response_text, labels, other_label)
    if return_raw:
        return {"raw_response": response_text, "parsed_type": parsed_type}
    return parsed_type


def classify_incidents(
    dataframe: pd.DataFrame,
    tokenizer,
    model,
    text_column: str,
    incident_types: list[str],
    prompt_builder: Callable[[str], str],
    progress_label: str,
) -> pd.DataFrame:
    if text_column not in dataframe.columns:
        raise ValueError(f"Колонка '{text_column}' не найдена. Есть: {list(dataframe.columns)}")

    result_df = dataframe.copy()
    result_df[TYPE_COLUMN] = [
        generate_incident_type(description, tokenizer, model, incident_types, prompt_builder)
        for description in tqdm(result_df[text_column], total=len(result_df), desc=progress_label)
    ]

    description_column = TEXT_COLUMN if TEXT_COLUMN in result_df.columns else text_column
    columns = [column for column in result_df.columns if column != TYPE_COLUMN]
    insert_at = columns.index(description_column) + 1
    columns.insert(insert_at, TYPE_COLUMN)
    result_df = result_df[columns]
    result_df = result_df.drop(columns=[LEMMATIZED_TEXT_COLUMN], errors="ignore")

    return result_df


def save_result_table(dataframe: pd.DataFrame, output_path: Path) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)

    if output_path.suffix == ".csv":
        dataframe.to_csv(output_path, index=False)
    elif output_path.suffix in {".xlsx", ".xls"}:
        dataframe.to_excel(output_path, index=False)
    else:
        dataframe.to_parquet(output_path, index=False)


def show_experiment_breakdowns(dataframe: pd.DataFrame) -> None:
    df = dataframe.copy()

    df["Дата"] = pd.to_datetime(df["Дата"], errors="coerce", dayfirst=True)
    df["Час"] = pd.to_datetime(df["Время"], format="%H:%M", errors="coerce").dt.hour
    df["День недели"] = df["Дата"].dt.day_name()

    incidents_per_address = (
        df.groupby("Адрес")
        .size()
        .sort_values(ascending=False)
        .reset_index(name="Количество")
    )
    print("=== Инциденты по адресам ===")
    display(incidents_per_address)

    incidents_per_weekday = (
        df.groupby("День недели")
        .size()
        .reindex(
            ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"],
            fill_value=0,
        )
        .reset_index(name="Количество")
    )
    print("\n=== Инциденты по дням недели ===")
    display(incidents_per_weekday)

    incidents_per_hour = (
        df.groupby("Час")
        .size()
        .reindex(range(24), fill_value=0)
        .reset_index(name="Количество")
    )
    print("\n=== Инциденты по часам ===")
    display(incidents_per_hour)

    address_incidents = df.groupby(["Адрес", TYPE_COLUMN]).size().unstack(fill_value=0)
    print("\n=== Детально по адресам ===")
    display(address_incidents)


In [70]:

tokenizer, model = load_generation_model(
    MODEL_NAME,
    cache_dir=MODEL_CACHE_DIR,
    local_files_only=LOCAL_FILES_ONLY,
)


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [71]:
print(torch.cuda.is_available())
print(next(model.parameters()).device)

True
cuda:0



## Эксперимент 1

Базовый сценарий с категориями `Стояк/Труба ГВС/ХВС`, плюс отдельная категория `Лифты`.


In [ ]:

EXPERIMENT_1_NAME = "base_plus_lifts"
EXPERIMENT_1_TYPES = [
    "Стояк ГВС",
    "Труба ГВС",
    "Стояк ХВС",
    "Труба ХВС",
    "Лифты",
]
EXPERIMENT_1_OUTPUT_PATH = ROOT / "research/data" / "incident_analysis_llm_base_plus_lifts.xlsx"
EXPERIMENT_1_PROMPT_INSTRUCTIONS = """Определи тип инцидента по тексту заявки ЖКХ.
Выбери ровно один тип из списка доступных типов.
Значение incident_type должно полностью совпадать с одним из доступных типов.
Не придумывай новые типы. Не добавляй к типу слова вроде "течь", "ремонт", "замена", "авария".

Правила:
1. Если заявка про лифт, выбери "Лифты".
2. Если явно указан стояк ГВС: стояк ГВС, ст. ГВС, ст ГВС, ст 29 ГВС, выбери "Стояк ГВС".
3. Если явно указан стояк ХВС: стояк ХВС, ст. ХВС, ст ХВС, ст 29 ХВС, выбери "Стояк ХВС".
4. Если указана труба, участок трубы или трубопровод ГВС без явного стояка, выбери "Труба ГВС".
5. Если указана труба, участок трубы или трубопровод ХВС без явного стояка, выбери "Труба ХВС".
6. Если невозможно выбрать один тип, выбери "Прочее".

Подсказки:
- "Течь стояка ГВС", "ремонт стояка ГВС", "замена стояка ГВС" -> "Стояк ГВС".
- "Течь стояка ХВС", "ремонт стояка ХВС", "замена стояка ХВС" -> "Стояк ХВС".
- "Течь трубы ГВС", "замена участка трубы ГВС", "ремонт трубопровода ГВС" -> "Труба ГВС".
- "Течь трубы ХВС", "замена участка трубы ХВС", "ремонт трубопровода ХВС" -> "Труба ХВС".
- "ст. 29 хгвс" или "стояк хгвс" -> "Стояк ГВС".
"""


In [74]:

experiment_1_df = classify_incidents(
    preprocessed_df,
    tokenizer=tokenizer,
    model=model,
    text_column=LLM_TEXT_COLUMN,
    incident_types=EXPERIMENT_1_TYPES,
    prompt_builder=partial(
        build_prompt,
        incident_types=EXPERIMENT_1_TYPES,
        prompt_instructions=EXPERIMENT_1_PROMPT_INSTRUCTIONS,
    ),
    progress_label=EXPERIMENT_1_NAME,
)
save_result_table(experiment_1_df, EXPERIMENT_1_OUTPUT_PATH)

print(f"Файл сохранен: {EXPERIMENT_1_OUTPUT_PATH.resolve()}")
display(experiment_1_df[[TEXT_COLUMN, TYPE_COLUMN]].head())
display(
    experiment_1_df[TYPE_COLUMN]
    .value_counts(dropna=False)
    .rename_axis(TYPE_COLUMN)
    .reset_index(name="count")
)


base_plus_lifts:   0%|          | 0/458 [00:00<?, ?it/s]

Файл сохранен: /home/fedor/Projects/building_maintenance_agents/research/data/incident_analysis_llm_base_plus_lifts.xlsx


,Описание,Тип инцидента
0,"СИЛЬНАЯ ТЕЧЬ СТОЯКА ГВС В ВАННОЙ, ОТКЛ. СТ. 29 ГВС В/З, ТРЕБ. ЗАМЕНА ОТСЕЧНОГО КРАНА, ВЫРВАЛО ШТОК\nГИЛЬДИЯ: Заменили два крана 1/2 на хгвс ст. 29 в/з\nЗапустили развоздушили",Прочее
1,течь с потолка-ТЕЧЬ С КРОВЛИ,Прочее
2,"ТЕЧЬ ПО СТЕНЕ В КОМНАТЕ +ТЕЧЕТ В КВАРТИРНОМ КОРИДОРЕ ,ТЕЧЬ СПУСКНИКА НА ТЕХ/ЭТ ,ЗАКРЫЛИ СПУСКНИК",Прочее
3,3 пар. пас. 1934 застревание,Прочее
4,"течь п/суш. (981) 822-05-22, течь соед. на ст. гвс н/з, откл. ст. гвс 18, ТРЕБ. РАЗБОР КОРОБА В КВ. 21 - РАЗОБРАЛИ, ЖДУТ 16.03.2026 В 10-00",Прочее


,Тип инцидента,count
0,Прочее,219
1,Лифты,151
2,Стояк ГВС,47
3,Труба ГВС,36
4,Стояк ХВС,3
5,Труба ХВС,2


In [75]:

show_experiment_breakdowns(experiment_1_df)


=== Инциденты по адресам ===


,Адрес,Количество
0,"г Санкт-Петербург, п Парголово, ул Фёдора Абрамова, д. 8 лит. А",34
1,"г Санкт-Петербург, п Парголово, ул Михаила Дудина, д. 25 корп. 1 лит. А",27
2,"г Санкт-Петербург, п Парголово, ул Фёдора Абрамова, д. 4 лит. А",27
3,"г Санкт-Петербург, п Парголово, ул Михаила Дудина, д. 25 корп. 2 лит. А",22
4,"г Санкт-Петербург, п Парголово, ул Фёдора Абрамова, д. 21 корп. 3 стр. 1",18
...,...,...
79,"г Санкт-Петербург, пр-кт Юнтоловский, д. 47 корп. 6 лит. А",1
80,"г Санкт-Петербург, пр-кт Юнтоловский, д. 55 корп. 2 стр. 1",1
81,"г Санкт-Петербург, ул Ивинская, д. 11 стр. 1",1
82,"г Санкт-Петербург, ул Ивинская, д. 17 стр. 1",1



=== Инциденты по дням недели ===


,День недели,Количество
0,Monday,62
1,Tuesday,59
2,Wednesday,75
3,Thursday,71
4,Friday,57
5,Saturday,44
6,Sunday,90



=== Инциденты по часам ===


,Час,Количество
0,0,6
1,1,4
2,2,3
3,3,3
4,4,0
5,5,1
6,6,6
7,7,12
8,8,15
9,9,26



=== Детально по адресам ===


Тип инцидента,Лифты,Прочее,Стояк ГВС,Стояк ХВС,Труба ГВС,Труба ХВС
Адрес,,,,,,
"г Санкт-Петербург, п Парголово, пр-д Толубеевский, д. 14 корп. 1 стр. 1",0,3,1,0,0,0
"г Санкт-Петербург, п Парголово, пр-д Толубеевский, д. 18 корп. 1 стр. 1",0,0,0,0,1,0
"г Санкт-Петербург, п Парголово, пр-д Толубеевский, д. 20 корп. 1 стр. 1",2,2,2,0,0,0
"г Санкт-Петербург, п Парголово, пр-д Толубеевский, д. 24 стр. 1",0,3,1,0,0,0
"г Санкт-Петербург, п Парголово, пр-д Толубеевский, д. 26 корп. 1 стр. 1",2,2,0,0,0,0
...,...,...,...,...,...,...
"г Санкт-Петербург, ул Ивинская, д. 17 стр. 1",0,1,0,0,0,0
"г Санкт-Петербург, ул Ивинская, д. 19 корп. 2 стр. 1",0,1,0,0,0,0
"г Санкт-Петербург, ул Ивинская, д. 7 стр. 1",1,1,0,0,0,0



## Эксперимент 2

Более детальный сценарий.


In [ ]:

EXPERIMENT_2_NAME = "detailed_pipes_plus_lifts"
EXPERIMENT_2_TYPES = [
    "Анализ труб",
    "Утечка радиатора",
    "Утечка стояка",
    "Замена трубы ГВС",
    "Замена трубы ХВС",
    "Замена ХВС на уровне тех этажа",
    "Замена главных стояков ГВС",
    "Замена розлива",
    "Лифты",
]
EXPERIMENT_2_OUTPUT_PATH = ROOT / "research/data" / "incident_analysis_llm_detailed_pipes_plus_lifts.xlsx"
EXPERIMENT_2_PROMPT_INSTRUCTIONS = """Определи тип инцидента по тексту заявки ЖКХ.
Выбери ровно один вариант из списка.
Если в тексте несколько действий, выбери основное событие или основную работу.

Подсказки по классам:
- Анализ труб: обследование, диагностика, осмотр, анализ труб без явной замены как основного действия.
- Утечка радиатора: течь радиатора, батареи или похожего отопительного прибора.
- Утечка стояка: течь стояка.
- Замена трубы ГВС: замена трубы или трубопровода горячего водоснабжения.
- Замена трубы ХВС: замена трубы или трубопровода холодного водоснабжения.
- Замена ХВС на уровне тех этажа: замена ХВС на техэтаже или техническом этаже.
- Замена главных стояков ГВС: замена именно главных стояков ГВС.
- Замена розлива: замена розлива.
- Лифты: лифт не работает, застревание, неисправность лифта, остановка лифта.
- Прочее: если ни один вариант не подходит."""


In [ ]:

experiment_2_df = classify_incidents(
    preprocessed_df,
    tokenizer=tokenizer,
    model=model,
    text_column=LLM_TEXT_COLUMN,
    incident_types=EXPERIMENT_2_TYPES,
    prompt_builder=partial(
        build_prompt,
        incident_types=EXPERIMENT_2_TYPES,
        prompt_instructions=EXPERIMENT_2_PROMPT_INSTRUCTIONS,
    ),
    progress_label=EXPERIMENT_2_NAME,
)
save_result_table(experiment_2_df, EXPERIMENT_2_OUTPUT_PATH)

print(f"Файл сохранен: {EXPERIMENT_2_OUTPUT_PATH.resolve()}")
display(experiment_2_df[[TEXT_COLUMN, TYPE_COLUMN]].head())
display(
    experiment_2_df[TYPE_COLUMN]
    .value_counts(dropna=False)
    .rename_axis(TYPE_COLUMN)
    .reset_index(name="count")
)


In [ ]:

show_experiment_breakdowns(experiment_2_df)
